# SD3.5 inpaint-EDIT LoRA — all-in-one (train → test → contact sheet)

Runs the whole edit flow in ONE session, **batch-safe for Kaggle "Save Version"**
(no kernel-restart cell), so nothing is lost to the between-session
`/kaggle/working` wipe:

1. setup (pinned deps, no restart)  2. SD3.5 access + build PIPE eval
3. train edit-LoRA on PIPE pairs    4. load-back the trained adapter
5. run edit on the eval set          6. metrics + contact sheet (manual check)

Learns: fill a masked region so the inserted object matches the photo. Background
is preserved 100% by hard-restore. Trains TWO artifacts (LoRA + input_adapter.pt).

**To run unattended:** set `SMOKE = False` in the train cell, then **Save Version
→ Save & Run All (Commit)**. Requires GPU + SD3.5 access (HF_TOKEN secret or a
mounted model dataset).

## 1. Setup — install pinned stack (batch-safe, no kernel restart)

Designed to run end-to-end via **Save Version**. The pip install runs before any
`transformers`/`diffusers` import, so a fresh batch kernel loads the pinned
versions directly (no `os._exit` restart, which would abort a batch run). A hard
version assert fails loudly if Kaggle pre-imported a wrong version.

In [ ]:
import subprocess, sys, os
from pathlib import Path

# IMPORTANT (batch-safe): do NOT import transformers/diffusers before this cell.
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

# diffusers 0.31.0 needs transformers<=4.46 (FLAX_WEIGHTS_NAME). Install the pinned
# stack first; a fresh batch kernel hasn't imported transformers yet, so these load.
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2','datasets>=2.20',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import transformers, diffusers, torch
print('transformers', transformers.__version__, '| diffusers', diffusers.__version__)
assert transformers.__version__ == '4.46.3', (
    f'transformers is {transformers.__version__}, expected 4.46.3. Kaggle pre-imported a '
    'newer build. Fix: in notebook Settings set "Environment -> Pin to original" OFF / use '
    'the latest env, OR run this cell, then Factory-reset & Run All. For Save Version, ensure '
    'no earlier cell imported transformers.')
from transformers.utils import FLAX_WEIGHTS_NAME  # symbol diffusers 0.31 needs
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU'
print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access (gated) + build the PIPE golden eval set

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local); print('local SD3.5 mount:', SD35_MODEL)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, ('SD3.5 is gated. Add a Kaggle secret HF_TOKEN (Read token + '
                      'agreed access at hf.co/stabilityai/stable-diffusion-3.5-medium) '
                      'or mount the model dataset.')
    from huggingface_hub import login; login(token=HF_TOKEN); print('HF login OK')

In [ ]:
from LoRA.data.build_eval_cases_pipe import run_build_pipe_eval
import json
WORK = Path('/kaggle/working/vin_lora')
EVAL = WORK/'eval'/'pipe_eval_v1'
EVAL_LIMIT = 12   # small for a fast contact sheet; raise for a fuller eval
if not (EVAL/'cases.jsonl').exists():
    run_build_pipe_eval(WORK, eval_set='pipe_eval_v1', split='test', person_only=True, limit=EVAL_LIMIT)
cases = [json.loads(l) for l in (EVAL/'cases.jsonl').read_text().splitlines() if l.strip()]
print(len(cases), 'eval cases')

## 3. Train the edit-LoRA on PIPE pairs

Smoke first (200 steps / 200 samples). For a real adapter set `SMOKE = False`
(uses the config: 1000 steps / 4000 samples — much longer).

In [ ]:
from LoRA.train.train_inpaint_edit import run_training
SMOKE = True
kw = dict(max_train_steps=200, num_train_samples=200) if SMOKE else {}
train = run_training(WORK, base_model_id=SD35_MODEL, hf_token=HF_TOKEN, **kw)
RUN_DIR = train['run_dir']
print('trained ->', RUN_DIR)
train['provenance']

## 4. Free training memory + load the trained adapter for inference

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print('free VRAM (MB):', round(torch.cuda.mem_get_info()[0]/1024**2, 1))

In [ ]:
from LoRA.inference.sd35_edit_runner import load_edit_runner_from_run
runner, prov = load_edit_runner_from_run(RUN_DIR, base_model_id=SD35_MODEL, hf_token=HF_TOKEN)
assert prov.get('requires_input_adapter')
print('loaded edit adapter from', RUN_DIR)

## 5. Run the edit on the eval set (background hard-restored)

In [ ]:
from PIL import Image
from LoRA.inference.inpaint_metrics import compute_case_metrics
RUN = WORK/'runs'/'edit_all_in_one'
(RUN/'images').mkdir(parents=True, exist_ok=True)
def case_prompt(c):
    return 'a photo of <vin_ped> pedestrian, ' + (c['prompt_fields'].get('instruction','') or 'a person')
runner.precompute_embeds([case_prompt(c) for c in cases])
rows = []
for c in cases:
    src = Image.open(EVAL/c['image_path']); msk = Image.open(EVAL/c['mask_path'])
    out = runner.edit(src, msk, case_prompt(c), seed=42, num_inference_steps=30)
    p = RUN/'images'/f"{c['case_id']}.png"; out.save(p)
    m = compute_case_metrics(EVAL/c['reference_path'], p, EVAL/c['mask_path'], c['expected_bbox_xyxy'], detector=None)
    m['case_id'] = c['case_id']; rows.append(m)
print('generated', len(rows), 'edits ->', RUN)

## 6. Contact sheet (manual check): source | mask | edit result

In [ ]:
from PIL import Image
import csv
with open(RUN/'edit_metrics.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r})); w.writeheader(); w.writerows(rows)
strips = []
for c in cases:
    src = Image.open(EVAL/c['image_path']).convert('RGB').resize((256,256))
    msk = Image.open(EVAL/c['mask_path']).convert('RGB').resize((256,256))
    res = Image.open(RUN/'images'/f"{c['case_id']}.png").convert('RGB').resize((256,256))
    s = Image.new('RGB',(768,256)); s.paste(src,(0,0)); s.paste(msk,(256,0)); s.paste(res,(512,0)); strips.append(s)
sheet = Image.new('RGB',(768,256*len(strips)))
for i,s in enumerate(strips): sheet.paste(s,(0,256*i))
sheet.save(RUN/'contact_sheet.png')
print('contact sheet ->', RUN/'contact_sheet.png   (cols: source | mask | edit)')
from IPython.display import Image as IPImage, display; display(IPImage(str(RUN/'contact_sheet.png')))

## 7. (optional) Save the adapter so it survives the session
Download this zip and upload it as a Kaggle Dataset to reuse without retraining.

In [ ]:
import shutil
z = shutil.make_archive(str(RUN_DIR), 'zip', str(RUN_DIR))
print('adapter+provenance zip ->', z)